# Animationen von Change

Ziel

Für jedes Land und jedes Jahr:

jährliche Änderung in GHG per capita und Vulnerability

Bewegungsvektor und Norm

Flag, ob es eine echte Verbesserung ist (beide Dimensionen ↓)


Output(animationen):
- **Globaler Kontext:** Animation aller Länder über alle Jahre im (GHG per capita, Vulnerability)-Raum zur Einordnung der Gesamtentwicklung.
- **Echte Verbesserungen:** Animation nur jener Jahre, in denen Länder gleichzeitig sinkende GHG-Emissionen pro Kopf und sinkende Vulnerability aufweisen.
- **Stabile Improvers:** Fokussierte Animation der Länder mit nachhaltiger und substanzieller Verbesserung über mehrere Jahre hinweg.



In [103]:
import numpy as np
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter






## Preprocessing


### Movement DataFrame: jährliche Dynamik pro Land

**Ziel**
Erzeuge aus dem Rohdatensatz eine **jährliche Bewegungsbeschreibung** pro Land, um Veränderungen in Emissionen und Vulnerability explizit analysieren zu können.

**Schritte**
1. Sortiere die Daten nach `ISO3` und `Year`, um die zeitliche Reihenfolge pro Land sicherzustellen.
2. Berechne jährliche Veränderungen pro Land:
   - `d_ghg`: Änderung der Emissionen pro Kopf gegenüber dem Vorjahr
   - `d_vuln`: Änderung der Vulnerability gegenüber dem Vorjahr
3. Interpretiere diese Änderungen als Bewegungsvektor im 2D-Raum.
4. Berechne die Bewegungsstärke als euklidische Norm:
   \[
   \sqrt{d\_ghg^2 + d\_vuln^2}
   \]
5. Markiere **echte Verbesserungen**:
   - `both_improved = True`, wenn beide Dimensionen abnehmen.

**Logik**


In [104]:



def build_movement_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Sortieren ist kritisch für korrekte Deltas
    df = df.sort_values(["ISO3", "Year"])

    # Jährliche Deltas pro Land
    df["d_ghg"] = df.groupby("ISO3")["GHG_per_capita"].diff()
    df["d_vuln"] = df.groupby("ISO3")["Vulnerability"].diff()

    # Bewegungsnorm
    df["movement_norm"] = np.sqrt(df["d_ghg"]**2 + df["d_vuln"]**2)

    # Strenge Verbesserung (beide Dimensionen besser)
    df["both_improved"] = (df["d_ghg"] < 0) & (df["d_vuln"] < 0)

    # Richtung explizit (optional, hilfreich später)
    df["dir_left"] = df["d_ghg"] < 0
    df["dir_down"] = df["d_vuln"] < 0

    return df




In [105]:

merged_df = pd.read_csv("data/merged_df.csv")
movement_df = build_movement_df(merged_df)
movement_df.head(10)

,ISO3,Country,Year,GHG_per_capita,ND_GAIN,Vulnerability,GDP_per_capita,d_ghg,d_vuln,movement_norm,both_improved,dir_left,dir_down
0,AFG,Afghanistan,1995,0.728879,34.783530,0.612511,813.550256,NaN,NaN,NaN,False,False,False
1,AFG,Afghanistan,1996,0.770651,34.775074,0.612679,813.550256,0.041772,0.000168,0.041773,False,False,False
2,AFG,Afghanistan,1997,0.805011,34.988812,0.612837,813.550256,0.034360,0.000158,0.034361,False,False,False
3,AFG,Afghanistan,1998,0.829508,35.293407,0.611179,813.550256,0.024496,-0.001659,0.024552,False,False,True
4,AFG,Afghanistan,1999,0.850790,35.177507,0.609828,813.550256,0.021282,-0.001351,0.021325,False,False,True
5,AFG,Afghanistan,2000,0.714894,35.065559,0.608398,813.550256,-0.135896,-0.001430,0.135904,True,True,True
6,AFG,Afghanistan,2001,0.602315,35.198269,0.606533,747.688045,-0.112579,-0.001865,0.112594,True,True,True
7,AFG,Afghanistan,2002,0.692841,35.335123,0.604601,926.507941,0.090525,-0.001931,0.090546,False,False,True
8,AFG,Afghanistan,2003,0.687519,35.542042,0.593451,966.962032,-0.005321,-0.011150,0.012355,True,True,True
9,AFG,Afghanistan,2004,0.663101,35.409770,0.595817,971.633503,-0.024418,0.002366,0.024533,False,True,False


### Ranking: Länder nach nachhaltiger Verbesserung

**Ziel**
Aggregiere die jährlichen Bewegungen zu **Länderkennzahlen**, um Länder nach Dauer und Stärke ihrer normativen Verbesserung zu vergleichen.

**Schritte**
1. Beschränke die Daten auf Jahre mit **echter Verbesserung**
   (`both_improved == True`).

2. Aggregiere pro Land:
   - `improvement_years`: Anzahl Jahre mit gleichzeitiger Verbesserung
   - `cumulative_movement`: Summe der Bewegungsstärken über diese Jahre
   - `avg_movement`: durchschnittliche jährliche Bewegungsstärke
   - `first_year`, `last_year`: Zeitraum der Verbesserungsphase

3. Sortiere Länder absteigend nach `cumulative_movement`.

**Logik**
- Es werden nur Jahre berücksichtigt, in denen sich ein Land **in beiden Dimensionen verbessert**.
- Die Kombination aus Anzahl der Jahre und Bewegungsstärke erfasst **nachhaltige statt kurzfristige** Verbesserungen.
- Die Sortierung priorisiert Länder mit der **grössten kumulativen Bewegung Richtung links unten**.
---

### Ranking: Länder nach nachhaltiger Verbesserung (mathematische Definition)

Sei für Land $i$ und Jahr $t$:

- $G_{i,t}$: GHG per capita
- $V_{i,t}$: Vulnerability

**Jährliche Änderungen**
$$
\Delta G_{i,t} = G_{i,t} - G_{i,t-1}
$$
$$
\Delta V_{i,t} = V_{i,t} - V_{i,t-1}
$$

**Strenge Verbesserung**
$$
I_{i,t} =
\begin{cases}
1, & \text{falls } \Delta G_{i,t} < 0 \;\land\; \Delta V_{i,t} < 0 \\
0, & \text{sonst}
\end{cases}
$$

**Bewegungsstärke (Norm)**
$$
\lVert \mathbf{d}_{i,t} \rVert
=
\sqrt{(\Delta G_{i,t})^2 + (\Delta V_{i,t})^2}
$$

**Anzahl Verbesserungsjahre**
$$
N_i = \sum_t I_{i,t}
$$

**Kumulative Verbesserung**
$$
S_i = \sum_{t:\, I_{i,t}=1} \lVert \mathbf{d}_{i,t} \rVert
$$

**Durchschnittliche jährliche Verbesserung**
$$
\bar{S}_i = \frac{S_i}{N_i}
$$

Diese Grössen bilden die Grundlage für das Länder-Ranking nach **Dauer** und **Stärke** der nachhaltigen Verbesserung.



In [106]:
def build_country_improvement_ranking(movement_df: pd.DataFrame) -> pd.DataFrame:
    df = movement_df.copy()

    # Nur echte Verbesserungsjahre zählen
    improved = df[df["both_improved"]].copy()

    ranking = (
        improved
        .groupby(["ISO3", "Country"])
        .agg(
            improvement_years=("Year", "count"),
            cumulative_movement=("movement_norm", "sum"),
            avg_movement=("movement_norm", "mean"),
            first_year=("Year", "min"),
            last_year=("Year", "max"),
        )
        .reset_index()
        .sort_values("cumulative_movement", ascending=False)
    )

    return ranking


In [107]:
country_ranking = build_country_improvement_ranking(movement_df)
country_ranking.head(10)


,ISO3,Country,improvement_years,cumulative_movement,avg_movement,first_year,last_year
130,PLW,Palau,9,227.130517,25.236724,1996,2016
136,QAT,Qatar,15,57.587028,3.839135,1998,2022
16,BHR,Bahrain,12,27.651835,2.304320,1997,2023
91,KWT,Kuwait,10,19.221255,1.922126,1998,2019
65,GNQ,Equatorial Guinea,13,17.373739,1.336441,2002,2023
24,BRN,Brunei,9,15.126264,1.680696,2002,2021
58,GAB,Gabon,11,13.699311,1.245392,1997,2021
53,EST,Estonia,6,11.949249,1.991542,1999,2023
100,LUX,Luxembourg,10,11.337282,1.133728,1996,2016
46,DNK,Denmark,14,11.166211,0.797586,1997,2023


### Filter: stabile Improvers (Länderauswahl)

**Ziel**
Identifiziere Länder mit nachhaltiger und substanzieller Verbesserung.

**Schritte**
1. Filtere Länder mit mindestens 5 Verbesserungsjahren:
   `improvement_years >= 5`

2. Filtere zusätzlich nach Stärke der Bewegung:
   `avg_movement > median(avg_movement)`

**Logik**
- Schritt 1 schliesst Zufall und Einjahres-Effekte aus.
- Schritt 2 schliesst triviale, kaum sichtbare Bewegungen aus.

→ Übrig bleiben Länder mit dauerhafter und relevanter Entwicklung.


In [108]:
stable_improvers = country_ranking[
    (country_ranking["improvement_years"] >= 5) &
    (country_ranking["avg_movement"] > country_ranking["avg_movement"].median())
]


### Trajektorien-DataFrame für Animation

**Ziel**
Erzeuge ein DataFrame mit **jährlichen Trajektorien** für genau jene Länder, die als *stabile Improvers* identifiziert wurden.

**Schritte**
1. Extrahiere die ISO-Codes der ausgewählten Länder aus `stable_improvers`.
2. Filtere `movement_df` auf diese Länder.
3. Sortiere nach `ISO3` und `Year`, um eine korrekte zeitliche Abfolge sicherzustellen.

**Logik**
- Das Ranking erfolgt auf **Länder-Ebene**, die Animation benötigt **Zeit-Ebene**.
- Dieser Schritt verbindet beides: Er überführt die Länderauswahl in **Jahr-für-Jahr-Trajektorien**.
- Das resultierende DataFrame bildet die direkte Grundlage für alle folgenden Animationen.


In [109]:
def build_trajectories_df(
    movement_df: pd.DataFrame,
    stable_improvers: pd.DataFrame
) -> pd.DataFrame:
    iso_keep = stable_improvers["ISO3"].unique()
    traj_df = movement_df[movement_df["ISO3"].isin(iso_keep)].copy()
    traj_df = traj_df.sort_values(["ISO3", "Year"])
    return traj_df


trajectories_df = build_trajectories_df(movement_df, stable_improvers)



## Animation 1: Globaler Kontext (alle Länder, alle Jahre)

**Ziel**
Zeige die Entwicklung des gesamten Systems über die Zeit im Raum
(GHG per capita, Vulnerability), um Spannweite, Cluster und generelle Trends einzuordnen.

**Schritte**
1. Verwende den Trajektorien-DataFrame mit **allen Ländern** und **allen Jahren**.
2. Erzeuge pro Jahr einen Frame und plotte die Länderpositionen als Scatter.
3. Halte die Achsen über alle Frames konstant (gleicher Ausschnitt), damit Bewegung vergleichbar bleibt.
4. Exportiere als MP4 für Slides.

**Logik**
Diese Animation liefert den Kontext: Sie zeigt nicht „Top Improvers“, sondern die globale Verteilung und deren zeitliche Verschiebung.
Alle späteren Fokus-Animationen sind Unterausschnitte oder Filter dieses globalen Bildes.


In [110]:


def animate_global_context(
    trajectories_df: pd.DataFrame,
    fps: int = 12,
    outfile: str = "anim_global_context.mp4",
    x_col: str = "GHG_per_capita",
    y_col: str = "Vulnerability",
    year_col: str = "Year",
) -> None:
    df = trajectories_df.copy()

    # Jahre
    years = sorted(df[year_col].dropna().unique())

    # Fixe Achsen
    x_min, x_max = df[x_col].min(), df[x_col].max()
    y_min, y_max = df[y_col].min(), df[y_col].max()


    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("GHG per capita")
    ax.set_ylabel("Vulnerability")
    title = ax.set_title("Global context")

    scat = ax.scatter([], [], s=18, alpha=0.55)

    def update(year: int):
        frame = df[df[year_col] == year]
        pts = frame[[x_col, y_col]].to_numpy()

        # set_offsets erwartet (n,2); für leere Frames safe-handling
        if pts.size == 0:
            scat.set_offsets(np.empty((0, 2)))
        else:
            scat.set_offsets(pts)

        title.set_text(f"Global context – {year}")
        return scat, title

    anim = FuncAnimation(
        fig,
        update,
        frames=years,
        interval=1000 / fps,
        blit=True,
    )

    anim.save(outfile, fps=fps)
    plt.close(fig)





In [69]:
# RUN
animate_global_context(
    trajectories_df=trajectories_df,  # alle Länder, alle Jahre
    fps=12,
    outfile="anim_global_context.mp4",
)

### Trail: Trajektorien sichtbar machen

**Ziel**
Mache Bewegungen nachvollziehbar, indem pro Jahr nicht nur die aktuelle Position, sondern auch die letzten $k$ Jahre als Spur angezeigt werden.

**Logik**
Ohne Trail sieht man nur „Punkte springen“. Mit Trail erkennt man Richtung, Geschwindigkeit und Stabilität der Trajektorien.


In [70]:



def animate_global_context_with_trail(
    trajectories_df: pd.DataFrame,
    fps: int = 3,
    outfile: str = "anim_global_context_trail.mp4",
    x_col: str = "GHG_per_capita",
    y_col: str = "Vulnerability",
    year_col: str = "Year",
    trail_years: int = 5,   # <- Spur-Länge (5 Jahre)
):
    df = trajectories_df.copy()
    years = sorted(df[year_col].dropna().unique())

    # Achsen: für globalen Kontext eher min/max oder sehr lockere Quantile
    x_min, x_max = df[x_col].min(), df[x_col].max()
    y_min, y_max = df[y_col].min(), df[y_col].max()

    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("GHG per capita")
    ax.set_ylabel("Vulnerability")
    title = ax.set_title("Global context")

    # Trail + Current
    trail_scat = ax.scatter([], [], s=10, alpha=0.15)  # Spur
    cur_scat = ax.scatter([], [], s=22, alpha=0.75)    # aktueller Punkt

    def update(year: int):
        # Jahre-Fenster für Trail
        y0 = year - trail_years
        trail = df[(df[year_col] >= y0) & (df[year_col] <= year)]
        cur = df[df[year_col] == year]

        trail_pts = trail[[x_col, y_col]].to_numpy()
        cur_pts = cur[[x_col, y_col]].to_numpy()

        trail_scat.set_offsets(trail_pts if trail_pts.size else np.empty((0, 2)))
        cur_scat.set_offsets(cur_pts if cur_pts.size else np.empty((0, 2)))

        title.set_text(f"Global context – {year} (trail: {trail_years}y)")
        return trail_scat, cur_scat, title

    anim = FuncAnimation(
        fig,
        update,
        frames=years,
        interval=1000 / fps,
        blit=True,
    )

    anim.save(outfile, fps=fps)
    plt.close(fig)





In [71]:
# RUN
animate_global_context_with_trail(
    trajectories_df=trajectories_df,
    fps=3,
    trail_years=5,
    outfile="anim_global_context_trail.mp4",
)

### Interpolation: Zwischenframes zwischen Jahreswerten (kontinuierliche Bewegung)

**Ziel**
Erzeuge zusätzliche Datenpunkte zwischen den Jahreswerten, damit die Animation nicht springt, sondern Bewegungen als kontinuierliche Trajektorien erscheinen.

**Schritte**
1. Sortiere pro Land nach `Year`.
2. Für jedes Land und jedes Jahrpaar $(t, t+1)$ berechne Zwischenpunkte mit linearem Anteil $\alpha \in [0,1]$.
3. Erzeuge pro Jahr `steps_per_year` Zwischenframes (z. B. 8 oder 10).
4. Speichere eine kontinuierliche Zeitvariable `t_cont = Year + alpha`.

**Logik**
- FPS steuert nur die Abspielgeschwindigkeit.
- Kontinuität entsteht erst durch mehr Frames zwischen Messpunkten.
- Lineare Interpolation bedeutet: Zwischenwerte sind „glatte Übergänge“ zwischen beobachteten Jahrespunkten (keine neuen Messdaten).


In [72]:



def build_interpolated_frames(
    trajectories_df: pd.DataFrame,
    steps_per_year: int = 8,
    x_col: str = "GHG_per_capita",
    y_col: str = "Vulnerability",
    year_col: str = "Year",
    id_col: str = "ISO3",
    keep_cols: tuple[str, ...] = ("ISO3", "Country"),
) -> pd.DataFrame:
    """
    Erzeugt Zwischenframes zwischen Jahrespunkten pro Land.
    Output enthält:
      - t_cont: kontinuierliche Zeit (Year + alpha)
      - frame: integer Frame-Index (für Animation)
      - x_col, y_col: interpolierte Positionen
    """
    df = trajectories_df.copy()
    df = df.sort_values([id_col, year_col])

    # alpha-Werte: z.B. bei steps_per_year=8 -> 0.0, 0.125, ..., 0.875
    alphas = np.linspace(0, 1, steps_per_year, endpoint=False)

    out = []
    for iso, g in df.groupby(id_col, sort=False):
        g = g.sort_values(year_col).reset_index(drop=True)

        # Wenn zu wenige Punkte: skip
        if len(g) < 2:
            continue

        # Paarweise Interpolation (t -> t+1)
        for i in range(len(g) - 1):
            y0 = int(g.loc[i, year_col])
            y1 = int(g.loc[i + 1, year_col])

            # Wenn Jahre Lücken haben, interpolieren wir trotzdem direkt von Punkt zu Punkt
            x0, x1 = float(g.loc[i, x_col]), float(g.loc[i + 1, x_col])
            v0, v1 = float(g.loc[i, y_col]), float(g.loc[i + 1, y_col])

            meta = {c: g.loc[i, c] for c in keep_cols if c in g.columns}

            for a in alphas:
                row = {
                    id_col: iso,
                    year_col: y0,           # Basisjahr (Start)
                    "alpha": float(a),      # Anteil zwischen y0 und y1
                    "t_cont": y0 + float(a),
                    x_col: (1 - a) * x0 + a * x1,
                    y_col: (1 - a) * v0 + a * v1,
                    **meta,
                }
                out.append(row)

        # letztes Jahres-„Ende“ ergänzen (alpha=1 am letzten Punkt)
        last = g.iloc[-1]
        meta_last = {c: last[c] for c in keep_cols if c in g.columns}
        out.append({
            id_col: iso,
            year_col: int(last[year_col]),
            "alpha": 1.0,
            "t_cont": float(last[year_col]),
            x_col: float(last[x_col]),
            y_col: float(last[y_col]),
            **meta_last,
        })

    interp_df = pd.DataFrame(out).sort_values([id_col, "t_cont"]).reset_index(drop=True)

    # globaler Frame-Index: sortierte unique t_cont
    t_vals = np.sort(interp_df["t_cont"].unique())
    t_to_frame = {t: i for i, t in enumerate(t_vals)}
    interp_df["frame"] = interp_df["t_cont"].map(t_to_frame).astype(int)

    return interp_df





In [73]:
# RUN (Beispiel)
interp_df = build_interpolated_frames(
    trajectories_df=trajectories_df,
    steps_per_year=8,
)

### Animation (global, kontinuierlich): interpolierte Frames + Trail

**Ziel**
Erzeuge eine globale Kontext-Animation mit **kontinuierlicher Bewegung**, indem zwischen Jahreswerten interpolierte Zwischenframes verwendet werden. Ein Trail macht die Trajektorien nachvollziehbar.

**Schritte**
1. Nutze `interp_df` als Frame-Basis (Spalten: `frame`, `GHG_per_capita`, `Vulnerability`).
2. Iteriere über `frame` statt über `Year`.
3. Zeichne pro Frame:
   - aktuelle Punkte (alle Länder)
   - Trail als Punkte aus den letzten `trail_frames`
4. Exportiere als MP4 (Slides).

**Logik**
- Kontinuität kommt von `steps_per_year` (mehr Zwischenframes).
- Clip-Länge steuerst du mit `fps` und `steps_per_year`.
- Trail erhöht Lesbarkeit der Bewegungsrichtung ohne Linien-Chaos.


In [74]:



def animate_global_context_continuous(
    interp_df: pd.DataFrame,
    fps: int = 10,
    outfile: str = "anim_global_context_continuous.mp4",
    x_col: str = "GHG_per_capita",
    y_col: str = "Vulnerability",
    frame_col: str = "frame",
    trail_frames: int = 40,   # ca. 40/steps_per_year Jahre Trail
) -> None:
    df = interp_df.copy()

    frames = sorted(df[frame_col].unique())

    # Für globalen Kontext: volle Spannweite (oder später optional Quantile)
    x_min, x_max = df[x_col].min(), df[x_col].max()
    y_min, y_max = df[y_col].min(), df[y_col].max()

    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("GHG per capita")
    ax.set_ylabel("Vulnerability")
    title = ax.set_title("Global context (continuous)")

    trail_scat = ax.scatter([], [], s=10, alpha=0.12)
    cur_scat = ax.scatter([], [], s=22, alpha=0.75)

    def update(f: int):
        # Trail-Fenster in Frames
        f0 = max(frames[0], f - trail_frames)
        trail = df[(df[frame_col] >= f0) & (df[frame_col] <= f)]
        cur = df[df[frame_col] == f]

        trail_pts = trail[[x_col, y_col]].to_numpy()
        cur_pts = cur[[x_col, y_col]].to_numpy()

        trail_scat.set_offsets(trail_pts if trail_pts.size else np.empty((0, 2)))
        cur_scat.set_offsets(cur_pts if cur_pts.size else np.empty((0, 2)))

        # optional: kontinuierliche Zeit im Titel (wenn vorhanden)
        if "t_cont" in cur.columns and len(cur) > 0:
            t_label = float(cur["t_cont"].iloc[0])
            title.set_text(f"Global context (continuous) – t={t_label:.2f}")
        else:
            title.set_text(f"Global context (continuous) – frame {f}")

        return trail_scat, cur_scat, title

    anim = FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=1000 / fps,
        blit=True,
    )

    anim.save(outfile, fps=fps)
    plt.close(fig)





In [75]:
# RUN
animate_global_context_continuous(
    interp_df=interp_df,
    fps=10,
    trail_frames=40,
    outfile="anim_global_context_continuous.mp4",
)

### Globaler Kontext (Animation): visuelles Clipping der GHG-Achse

**Ziel**
Erhöhe die Lesbarkeit der globalen Animation, indem extreme Ausreisser in
*GHG per capita* visuell begrenzt werden.

**Umsetzung**
- Setze die x-Achse fix auf `0–60` t CO₂/cap.
- **Nur Visualisierung**: Die Daten bleiben unverändert.

**Logik**
Eine kleine Anzahl extrem hoher Pro-Kopf-Emissionen streckt die Achse und drängt den Grossteil der Länder zusammen.
Das Clipping ist ein **Design-Zoom**, kein Eingriff in die Analyse.

**Hinweis (für Slides)**
„Countries above 60 t CO₂/cap are truncated for readability.“


In [76]:
def animate_global_context_continuous_clipped(
    interp_df: pd.DataFrame,
    fps: int = 10,
    outfile: str = "anim_global_context_continuous_clipped.mp4",
    x_col: str = "GHG_per_capita",
    y_col: str = "Vulnerability",
    frame_col: str = "frame",
    trail_frames: int = 40,
):
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation

    df = interp_df.copy()
    frames = sorted(df[frame_col].unique())

    # Y-Achse: volle Spannweite + kleines Padding
    y_min, y_max = df[y_col].min(), df[y_col].max()
    y_pad = 0.05 * (y_max - y_min)

    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    ax.set_xlim(0, 60)  # <<< visuelles Clipping
    ax.set_ylim(y_min - y_pad, y_max + y_pad)
    ax.set_xlabel("GHG per capita")
    ax.set_ylabel("Vulnerability")
    title = ax.set_title("Global context (continuous, clipped)")

    trail_scat = ax.scatter([], [], s=10, alpha=0.12)
    cur_scat = ax.scatter([], [], s=22, alpha=0.75)

    def update(f: int):
        f0 = max(frames[0], f - trail_frames)
        trail = df[(df[frame_col] >= f0) & (df[frame_col] <= f)]
        cur = df[df[frame_col] == f]

        trail_pts = trail[[x_col, y_col]].to_numpy()
        cur_pts = cur[[x_col, y_col]].to_numpy()

        trail_scat.set_offsets(trail_pts if trail_pts.size else np.empty((0, 2)))
        cur_scat.set_offsets(cur_pts if cur_pts.size else np.empty((0, 2)))

        if "t_cont" in cur.columns and len(cur) > 0:
            title.set_text(f"Global context (continuous) – t={cur['t_cont'].iloc[0]:.2f}")
        return trail_scat, cur_scat, title

    anim = FuncAnimation(fig, update, frames=frames, interval=1000 / fps, blit=True)
    anim.save(outfile, fps=fps)
    plt.close(fig)




In [77]:
# RUN
animate_global_context_continuous_clipped(
    interp_df=interp_df,
    fps=10,
    trail_frames=40,
    outfile="anim_global_context_continuous_clipped.mp4",
)

### Export: höhere Videoauflösung für Slides

**Ziel**
Erzeuge eine MP4 mit höherer visueller Qualität (weniger Pixeligkeit) für Präsentationsfolien.

**Schritte**
1. Erhöhe die Figurgrösse (`figsize`) für mehr Pixel.
2. Setze eine höhere DPI beim Speichern (`savefig.dpi`).
3. Verwende beim MP4-Export einen höheren Bitrate/Codec-Parameter (FFmpeg).

**Logik**
Die sichtbare Qualität wird primär durch **Pixelanzahl pro Frame** und **Kompressionsbitrate** bestimmt.
Für Slides lohnt sich ein höheres DPI/figsize + höhere Bitrate.


In [78]:



def animate_global_context_continuous_clipped_hd(
    interp_df: pd.DataFrame,
    fps: int = 10,
    outfile: str = "anim_global_context_continuous_clipped_HD.mp4",
    x_col: str = "GHG_per_capita",
    y_col: str = "Vulnerability",
    frame_col: str = "frame",
    trail_frames: int = 40,
    figsize: tuple[float, float] = (12, 9),   # größer = mehr Pixel
    dpi: int = 200,                           # höher = schärfer
    bitrate: int = 8000,                      # höher = weniger Kompression
):
    df = interp_df.copy()
    frames = sorted(df[frame_col].unique())

    # Y-Achse: volle Spannweite + Padding
    y_min, y_max = df[y_col].min(), df[y_col].max()
    y_pad = 0.05 * (y_max - y_min)

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(0, 60)  # visuelles Clipping
    ax.set_ylim(y_min - y_pad, y_max + y_pad)
    ax.set_xlabel("GHG per capita")
    ax.set_ylabel("Vulnerability")
    title = ax.set_title("Global context (continuous, clipped)")

    # stärkere Trennung current vs trail (ohne Farbwechsel)
    trail_scat = ax.scatter([], [], s=8, alpha=0.10)
    cur_scat = ax.scatter([], [], s=28, alpha=0.85)

    def update(f: int):
        f0 = max(frames[0], f - trail_frames)
        trail = df[(df[frame_col] >= f0) & (df[frame_col] <= f)]
        cur = df[df[frame_col] == f]

        trail_pts = trail[[x_col, y_col]].to_numpy()
        cur_pts = cur[[x_col, y_col]].to_numpy()

        trail_scat.set_offsets(trail_pts if trail_pts.size else np.empty((0, 2)))
        cur_scat.set_offsets(cur_pts if cur_pts.size else np.empty((0, 2)))

        if "t_cont" in cur.columns and len(cur) > 0:
            title.set_text(f"Global context (continuous) – t={cur['t_cont'].iloc[0]:.2f}")
        return trail_scat, cur_scat, title

    anim = FuncAnimation(fig, update, frames=frames, interval=1000 / fps, blit=True)

    writer = FFMpegWriter(
        fps=fps,
        bitrate=bitrate,
        codec="libx264",
        extra_args=["-pix_fmt", "yuv420p"],
    )
    anim.save(outfile, writer=writer, dpi=dpi)
    plt.close(fig)




In [79]:
# RUN
animate_global_context_continuous_clipped_hd(
    interp_df=interp_df,
    fps=10,
    trail_frames=40,
    outfile="anim_global_context_continuous_clipped_HD.mp4",
    figsize=(12, 9),
    dpi=200,
    bitrate=8000,
)

### Animation (global, kontinuierlich, HD – nicht geclippt)

**Ziel**
Exportiere die globale Kontext-Animation in **hoher Auflösung**, ohne visuelles Clipping der GHG-Achse, sodass die **volle Spannweite der Daten** sichtbar bleibt.

**Schritte**
1. Verwende interpolierte Frames (`interp_df`) mit Trail.
2. Setze die Achsen auf **Min/Max der Daten** (kein Zoom).
3. Erhöhe `figsize`, `dpi` und `bitrate` für scharfe MP4-Ausgabe.

**Logik**
Die analytische Darstellung bleibt unverändert.
Die höhere Auflösung reduziert Pixeligkeit in Slides, ohne die Aussage zu verändern.


In [80]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter


def animate_global_context_continuous_hd(
    interp_df: pd.DataFrame,
    fps: int = 10,
    outfile: str = "anim_global_context_continuous_HD.mp4",
    x_col: str = "GHG_per_capita",
    y_col: str = "Vulnerability",
    frame_col: str = "frame",
    trail_frames: int = 40,
    figsize: tuple[float, float] = (12, 9),  # HD: mehr Pixel
    dpi: int = 200,                          # HD: schärfer
    bitrate: int = 8000,                     # HD: weniger Kompression
) -> None:
    df = interp_df.copy()
    frames = sorted(df[frame_col].unique())

    # Volle Spannweite (nicht geclippt)
    x_min, x_max = df[x_col].min(), df[x_col].max()
    y_min, y_max = df[y_col].min(), df[y_col].max()

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("GHG per capita")
    ax.set_ylabel("Vulnerability")
    title = ax.set_title("Global context (continuous)")

    # klare Trennung: Trail vs. aktueller Punkt
    trail_scat = ax.scatter([], [], s=8, alpha=0.10)
    cur_scat = ax.scatter([], [], s=28, alpha=0.85)

    def update(f: int):
        f0 = max(frames[0], f - trail_frames)
        trail = df[(df[frame_col] >= f0) & (df[frame_col] <= f)]
        cur = df[df[frame_col] == f]

        trail_pts = trail[[x_col, y_col]].to_numpy()
        cur_pts = cur[[x_col, y_col]].to_numpy()

        trail_scat.set_offsets(trail_pts if trail_pts.size else np.empty((0, 2)))
        cur_scat.set_offsets(cur_pts if cur_pts.size else np.empty((0, 2)))

        if "t_cont" in cur.columns and len(cur) > 0:
            title.set_text(f"Global context (continuous) – t={cur['t_cont'].iloc[0]:.2f}")
        else:
            title.set_text(f"Global context (continuous) – frame {f}")

        return trail_scat, cur_scat, title

    anim = FuncAnimation(fig, update, frames=frames, interval=1000 / fps, blit=True)

    writer = FFMpegWriter(
        fps=fps,
        bitrate=bitrate,
        codec="libx264",
        extra_args=["-pix_fmt", "yuv420p"],
    )
    anim.save(outfile, writer=writer, dpi=dpi)
    plt.close(fig)





In [81]:
# RUN
animate_global_context_continuous_hd(
    interp_df=interp_df,
    fps=10,
    trail_frames=40,
    outfile="anim_global_context_continuous_HD.mp4",
)

## Animation 2: Nur positive Entwicklungen (HD, kontinuierlich)

**Ziel**
Visualisiere ausschliesslich **echte Verbesserungen**: Zeitabschnitte, in denen Länder gleichzeitig
- sinkende GHG pro Kopf **und**
- sinkende Vulnerability
aufweisen.

**Schritte**
1. Filtere interpolierte Frames auf **positive Bewegungssegmente**.
2. Zeige pro Frame nur jene Länder, die sich im aktuellen Intervall **normativ positiv** bewegen.
3. Nutze kontinuierliche Frames + Trail für nachvollziehbare Trajektorien.
4. Exportiere als **HD-MP4** (Slides).

**Logik**
- Die Selektion ist die Aussage: Es werden **keine Trade-offs** gezeigt.
- Kontinuität entsteht durch Interpolation; Lesbarkeit durch Trail.
- Achsen bleiben unverändert (volle Spannweite), damit die Darstellung mit dem globalen Kontext vergleichbar ist.


In [82]:
countries_animation_2 = (
    movement_df
    .loc[movement_df["both_improved"], ["ISO3", "Country"]]
    .drop_duplicates()
    .sort_values("Country")
    .reset_index(drop=True)
)

countries_animation_2


,ISO3,Country
0,AFG,Afghanistan
1,ALB,Albania
2,DZA,Algeria
3,AGO,Angola
4,ATG,Antigua and Barbuda
...,...,...
176,VEN,Venezuela
177,VNM,Viet Nam
178,YEM,Yemen
179,ZMB,Zambia


In [83]:


def animate_positive_trends_continuous_hd(
    interp_df: pd.DataFrame,
    movement_df: pd.DataFrame,
    fps: int = 10,
    outfile: str = "anim_positive_trends_continuous_HD.mp4",
    x_col: str = "GHG_per_capita",
    y_col: str = "Vulnerability",
    frame_col: str = "frame",
    year_col: str = "Year",
    id_col: str = "ISO3",
    trail_frames: int = 40,
    figsize: tuple[float, float] = (12, 9),
    dpi: int = 200,
    bitrate: int = 8000,
) -> None:
    """
    Zeigt nur positive Entwicklungen:
    Für jedes Jahresintervall (t -> t+1) werden nur Länder angezeigt,
    die im movement_df both_improved == True haben.
    Die interpolierten Frames innerhalb dieses Intervalls werden gezeigt,
    alle anderen ausgeblendet.
    """
    # --- 1) Positive Jahresintervalle bestimmen ---
    pos_years = (
        movement_df[movement_df["both_improved"]]
        [[id_col, year_col]]
        .assign(year_end=lambda d: d[year_col] + 1)
    )

    # Join: markiere interpolierte Frames, die in ein positives Intervall fallen
    df = interp_df.copy()
    df = df.merge(
        pos_years,
        on=id_col,
        how="left",
        suffixes=("", "_pos"),
    )

    # Ein interpolierter Frame gehört zum positiven Intervall,
    # wenn seine kontinuierliche Zeit zwischen Year und Year+1 liegt
    df["is_positive_frame"] = (
        (df["t_cont"] >= df[year_col]) &
        (df["t_cont"] < df["year_end"])
    )

    df = df[df["is_positive_frame"]].copy()

    # --- 2) Frames & Achsen ---
    frames = sorted(df[frame_col].unique())
    x_min, x_max = df[x_col].min(), df[x_col].max()
    y_min, y_max = df[y_col].min(), df[y_col].max()

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("GHG per capita")
    ax.set_ylabel("Vulnerability")
    title = ax.set_title("Positive developments only (continuous)")

    trail_scat = ax.scatter([], [], s=8, alpha=0.10)
    cur_scat = ax.scatter([], [], s=28, alpha=0.85)

    # --- 3) Update ---
    def update(f: int):
        f0 = max(frames[0], f - trail_frames)
        trail = df[(df[frame_col] >= f0) & (df[frame_col] <= f)]
        cur = df[df[frame_col] == f]

        trail_pts = trail[[x_col, y_col]].to_numpy()
        cur_pts = cur[[x_col, y_col]].to_numpy()

        trail_scat.set_offsets(trail_pts if trail_pts.size else np.empty((0, 2)))
        cur_scat.set_offsets(cur_pts if cur_pts.size else np.empty((0, 2)))

        if "t_cont" in cur.columns and len(cur) > 0:
            title.set_text(f"Positive developments only – t={cur['t_cont'].iloc[0]:.2f}")
        return trail_scat, cur_scat, title

    anim = FuncAnimation(fig, update, frames=frames, interval=1000 / fps, blit=True)

    writer = FFMpegWriter(
        fps=fps,
        bitrate=bitrate,
        codec="libx264",
        extra_args=["-pix_fmt", "yuv420p"],
    )
    anim.save(outfile, writer=writer, dpi=dpi)
    plt.close(fig)


# RUN
animate_positive_trends_continuous_hd(
    interp_df=interp_df,
    movement_df=movement_df,
    fps=10,
    trail_frames=40,
    outfile="anim_positive_trends_continuous_HD.mp4",
)


### Patch: Positive-only Animation korrekt filtern (ISO3 × Startjahr)

**Ziel**
Stelle sicher, dass wirklich nur jene interpolierten Frames gezeigt werden, die zu einem Jahresintervall gehören, in dem ein Land **gleichzeitig**:
- $\Delta$GHG < 0 und
- $\Delta$Vulnerability < 0

**Schritte**
1. Erzeuge aus `movement_df` eine Lookup-Tabelle `pos_lookup` mit `(ISO3, Year) -> both_improved`.
2. Nutze in `interp_df` die Spalte `Year` als **Startjahr des Interpolationssegments** (Intervall $[Year, Year+1)$).
3. Merge `interp_df` mit `pos_lookup` auf `(ISO3, Year)`.
4. Filtere strikt auf `both_improved == True`.

**Logik**
Der vorige Ansatz konnte Frames „durchlassen“, weil Jahr-Informationen aus Interpolation und Movement nicht eindeutig gekoppelt waren.
Mit `(ISO3, Year)` als Schlüssel ist jedes interpolierte Segment exakt dem Jahresdelta zugeordnet.


In [84]:
countries_animation_3 = (
    stable_improvers
    [["ISO3", "Country", "improvement_years", "avg_movement"]]
    .sort_values("Country")
    .reset_index(drop=True)
)

countries_animation_3


,ISO3,Country,improvement_years,avg_movement
0,DZA,Algeria,7,0.192188
1,ATG,Antigua and Barbuda,5,0.301502
2,ARG,Argentina,5,0.296945
3,ARM,Armenia,6,0.198904
4,AUS,Australia,9,0.543057
...,...,...,...,...
79,GBR,United Kingdom,10,0.366763
80,USA,United States,10,0.674204
81,URY,Uruguay,11,0.337846
82,UZB,Uzbekistan,7,0.316244


In [85]:


def animate_positive_trends_continuous_hd_strict(
    interp_df: pd.DataFrame,
    movement_df: pd.DataFrame,
    fps: int = 10,
    outfile: str = "anim_positive_trends_continuous_HD_STRICT.mp4",
    x_col: str = "GHG_per_capita",
    y_col: str = "Vulnerability",
    frame_col: str = "frame",
    year_col: str = "Year",
    id_col: str = "ISO3",
    trail_frames: int = 40,
    figsize: tuple[float, float] = (12, 9),
    dpi: int = 200,
    bitrate: int = 8000,
) -> None:
    """
    Positive-only, strikt:
    Ein interpolierter Frame gehört zum Jahresintervall [Year, Year+1) des Landes.
    Er wird nur gezeigt, wenn movement_df für genau dieses (ISO3, Year) both_improved == True hat.
    """
    # 1) Lookup: (ISO3, Year) -> both_improved
    pos_lookup = (
        movement_df[[id_col, year_col, "both_improved"]]
        .dropna(subset=[id_col, year_col])
        .copy()
    )

    # 2) Interpolierte Frames mit Lookup verbinden
    df = interp_df.copy()
    df = df.merge(pos_lookup, on=[id_col, year_col], how="left")

    # 3) Strikter Filter
    df = df[df["both_improved"] == True].copy()

    # Edge case: falls nach Filter leer
    if df.empty:
        raise ValueError("Nach strict positive-only Filter ist df leer. Prüfe movement_df['both_improved'] und Years.")

    # 4) Frames & Achsen (volle Spannweite, nicht geclippt)
    frames = sorted(df[frame_col].unique())
    x_min, x_max = df[x_col].min(), df[x_col].max()
    y_min, y_max = df[y_col].min(), df[y_col].max()

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("GHG per capita")
    ax.set_ylabel("Vulnerability")
    title = ax.set_title("Positive developments only (continuous, strict)")

    trail_scat = ax.scatter([], [], s=8, alpha=0.10)
    cur_scat = ax.scatter([], [], s=28, alpha=0.85)

    def update(f: int):
        f0 = max(frames[0], f - trail_frames)
        trail = df[(df[frame_col] >= f0) & (df[frame_col] <= f)]
        cur = df[df[frame_col] == f]

        trail_pts = trail[[x_col, y_col]].to_numpy()
        cur_pts = cur[[x_col, y_col]].to_numpy()

        trail_scat.set_offsets(trail_pts if trail_pts.size else np.empty((0, 2)))
        cur_scat.set_offsets(cur_pts if cur_pts.size else np.empty((0, 2)))

        if "t_cont" in cur.columns and len(cur) > 0:
            title.set_text(f"Positive developments only – t={cur['t_cont'].iloc[0]:.2f}")
        else:
            title.set_text(f"Positive developments only – frame {f}")

        return trail_scat, cur_scat, title

    anim = FuncAnimation(fig, update, frames=frames, interval=1000 / fps, blit=True)

    writer = FFMpegWriter(
        fps=fps,
        bitrate=bitrate,
        codec="libx264",
        extra_args=["-pix_fmt", "yuv420p"],
    )
    anim.save(outfile, writer=writer, dpi=dpi)
    plt.close(fig)


# RUN
animate_positive_trends_continuous_hd_strict(
    interp_df=interp_df,
    movement_df=movement_df,
    fps=10,
    trail_frames=40,
    outfile="anim_positive_trends_continuous_HD_STRICT.mp4",
)


## Animation 3: Stabile Improvers (HD, kontinuierlich, Fokus)

**Ziel**
Visualisiere die Trajektorien jener Länder, die im Ranking als **stabile Improvers - mindestens 5 Jahre** identifiziert wurden, damit ihre Bewegungen über die Zeit klar vergleichbar werden.

**Schritte**
1. Nimm die ISO3-Liste aus `stable_improvers`.
2. Filtere `interp_df` auf diese Länder (kontinuierliche Frames bleiben erhalten).
3. Animieren wie zuvor (Trail + aktueller Punkt), aber nur für dieses Subset.
4. Exportiere als HD-MP4 für Slides.

**Logik**
- Globaler Kontext zeigt Verteilung; stabile Improvers zeigen die Story.
- Durch Subset-Filter wird die Bewegung lesbar, ohne dass Ausreisser die Skala dominieren.
- Die Auswahl (Ranking + Filter) ist die inhaltliche Aussage; die Animation ist die Visualisierung davon.


In [86]:
# Animation 3: Stabile Improvers (HD, kontinuierlich, Fokus)
# Filter (bereits in stable_improvers enthalten):
# - improvement_years >= 5
# - avg_movement > median(avg_movement)

countries_animation_3 = (
    stable_improvers
    [["ISO3", "Country", "improvement_years", "avg_movement"]]
    .sort_values("Country")
    .reset_index(drop=True)
)

countries_animation_3


,ISO3,Country,improvement_years,avg_movement
0,DZA,Algeria,7,0.192188
1,ATG,Antigua and Barbuda,5,0.301502
2,ARG,Argentina,5,0.296945
3,ARM,Armenia,6,0.198904
4,AUS,Australia,9,0.543057
...,...,...,...,...
79,GBR,United Kingdom,10,0.366763
80,USA,United States,10,0.674204
81,URY,Uruguay,11,0.337846
82,UZB,Uzbekistan,7,0.316244


In [93]:

def animate_stable_improvers_continuous_hd(
    interp_df: pd.DataFrame,
    stable_improvers: pd.DataFrame,
    fps: int = 10,
    outfile: str = "anim_stable_improvers_continuous_HD.mp4",
    x_col: str = "GHG_per_capita",
    y_col: str = "Vulnerability",
    frame_col: str = "frame",
    id_col: str = "ISO3",
    trail_frames: int = 60,                 # etwas länger wirkt hier gut
    figsize: tuple[float, float] = (12, 9),
    dpi: int = 200,
    bitrate: int = 8000,
    pad_frac: float = 0.06,                 # etwas Luft um die Punkte
) -> None:
    """
    Fokus-Animation für stabile Improvers (Subset von Ländern).
    HD-Export, kontinuierlich (interp_df), Trail + aktueller Punkt.
    """
    iso_list = stable_improvers[id_col].dropna().unique().tolist()
    df = interp_df[interp_df[id_col].isin(iso_list)].copy()

    if df.empty:
        raise ValueError("Subset für stabile Improvers ist leer. Prüfe stable_improvers['ISO3'] und interp_df['ISO3'].")

    frames = sorted(df[frame_col].unique())

    # Achsen passend zum Subset (Fokus), mit Padding
    x_min, x_max = df[x_col].min(), df[x_col].max()
    y_min, y_max = df[y_col].min(), df[y_col].max()
    x_pad = pad_frac * (x_max - x_min) if x_max > x_min else 1.0
    y_pad = pad_frac * (y_max - y_min) if y_max > y_min else 0.01

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(x_min - x_pad, x_max + x_pad)
    ax.set_ylim(y_min - y_pad, y_max + y_pad)
    ax.set_xlabel("GHG per capita")
    ax.set_ylabel("Vulnerability")
    title = ax.set_title("Stable improvers (continuous)")

    trail_scat = ax.scatter([], [], s=10, alpha=0.12)
    cur_scat = ax.scatter([], [], s=38, alpha=0.90)

    def update(f: int):
        f0 = max(frames[0], f - trail_frames)
        trail = df[(df[frame_col] >= f0) & (df[frame_col] <= f)]
        cur = df[df[frame_col] == f]

        trail_pts = trail[[x_col, y_col]].to_numpy()
        cur_pts = cur[[x_col, y_col]].to_numpy()

        trail_scat.set_offsets(trail_pts if trail_pts.size else np.empty((0, 2)))
        cur_scat.set_offsets(cur_pts if cur_pts.size else np.empty((0, 2)))

        if "t_cont" in cur.columns and len(cur) > 0:
            title.set_text(f"Stable improvers – t={cur['t_cont'].iloc[0]:.2f}")
        return trail_scat, cur_scat, title

    anim = FuncAnimation(fig, update, frames=frames, interval=1000 / fps, blit=True)

    writer = FFMpegWriter(
        fps=fps,
        bitrate=bitrate,
        codec="libx264",
        extra_args=["-pix_fmt", "yuv420p"],
    )
    anim.save(outfile, writer=writer, dpi=dpi)
    plt.close(fig)


# RUN
animate_stable_improvers_continuous_hd(
    interp_df=interp_df,
    stable_improvers=stable_improvers,
    fps=10,
    trail_frames=60,
    outfile="anim_stable_improvers_continuous_HD.mp4",
)


### Kontinuitäts-Filter: minimale Laenge einer zusammenhaengenden Verbesserungsphase

**Ziel**
Statt "mindestens X gute Jahre irgendwo" filtern wir jetzt nach **echten Phasen**: ein Land soll ueber mehrere **aufeinanderfolgende Jahre** gleichzeitig
- $\\Delta$GHG < 0 und
- $\\Delta$Vulnerability < 0
haben.

**Vorschlag fuer die Mindestlaenge**
**4 Jahre am Stueck**.

**Warum 4 (Pro/Contra kurz)**
- **Pro:** stark genug, um Zufall/Einjahres-Effekte zu reduzieren; aber nicht so streng, dass fast alles rausfaellt.
- **Contra:** fuer sehr volatile Laender kann 4 zu streng sein, obwohl der Netto-Trend positiv ist.

**Praxisregel**
- Setze `min_streak_years = 4` als Default.
- Fuehre spaeter einen Sensitivity-Check mit 3 und 5 durch (nur als Robustheitscheck).


In [97]:
def build_continuity_ranking(
    movement_df: pd.DataFrame,
    min_streak_years: int = 4,
) -> pd.DataFrame:
    """
    Bestimmt pro Land die laengste zusammenhaengende Phase von Jahren,
    in denen both_improved == True gilt.
    """

    df = movement_df.copy()
    df = df.sort_values(["ISO3", "Year"])

    # Vorjahr-Infos
    df["prev_year"] = df.groupby("ISO3")["Year"].shift(1)
    df["prev_flag"] = df.groupby("ISO3")["both_improved"].shift(1)

    # Neue Streak, wenn:
    # - kein both_improved
    # - Unterbruch im Jahr
    # - oder Flag wechselt
    new_streak = (
        (df["both_improved"] != True) |
        (df["prev_flag"] != True) |
        (df["Year"] != df["prev_year"] + 1)
    )

    df["streak_id"] = new_streak.groupby(df["ISO3"]).cumsum()

    # Nur echte Verbesserungsjahre
    df_pos = df[df["both_improved"]].copy()

    # Streak-Laengen
    streaks = (
        df_pos
        .groupby(["ISO3", "Country", "streak_id"])
        .size()
        .reset_index(name="streak_len")
    )

    # Laengste Streak pro Land
    continuity_ranking = (
        streaks
        .groupby(["ISO3", "Country"])["streak_len"]
        .max()
        .reset_index(name="longest_streak")
        .sort_values("longest_streak", ascending=False)
        .reset_index(drop=True)
    )

    return continuity_ranking


In [98]:
min_streak_years = 4

continuity_ranking = build_continuity_ranking(
    movement_df=movement_df,
    min_streak_years=min_streak_years
)

countries_continuity = (
    continuity_ranking
    .loc[continuity_ranking["longest_streak"] >= min_streak_years,
         ["ISO3", "Country", "longest_streak"]]
    .sort_values(["longest_streak", "Country"], ascending=[False, True])
    .reset_index(drop=True)
)

countries_continuity


,ISO3,Country,longest_streak
0,SOM,Somalia,19
1,QAT,Qatar,7
2,POL,Poland,6
3,GNQ,Equatorial Guinea,5
4,ERI,Eritrea,5
5,FRA,France,5
6,GRC,Greece,5
7,NZL,New Zealand,5
8,NGA,Nigeria,5
9,PRY,Paraguay,5


In [99]:

def animate_continuity_improvers(
    interp_df: pd.DataFrame,
    countries_continuity: pd.DataFrame,
    highlight_iso3: list[str] = ["SOM", "QAT", "POL"],
    fps: int = 12,
    trail_frames: int = 40,
    outfile: str = "anim_continuity_improvers_HD.mp4",
):
    # --- Filter: nur Kontinuitäts-Improvers ---
    iso_keep = set(countries_continuity["ISO3"])
    df = interp_df[interp_df["ISO3"].isin(iso_keep)].copy()

    frames = sorted(df["frame"].unique())

    # Achsen fix (global konsistent)
    x_min, x_max = df["GHG_per_capita"].min(), df["GHG_per_capita"].max()
    y_min, y_max = df["Vulnerability"].min(), df["Vulnerability"].max()

    fig, ax = plt.subplots(figsize=(10, 7), dpi=150)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("GHG per capita")
    ax.set_ylabel("Vulnerability")
    title = ax.set_title("Continuity improvers (continuous)")

    # Scatter
    trail_scat = ax.scatter([], [], s=10, alpha=0.12, color="#1f77b4")
    cur_scat = ax.scatter([], [], s=26, alpha=0.85, color="#1f77b4")

    # Highlight Scatter
    hi_scat = ax.scatter([], [], s=50, color="#d62728", zorder=5)

    # Text-Labels (persistent)
    label_texts = {
        iso: ax.text(
            0, 0, "",
            fontsize=9,
            color="#d62728",
            ha="left",
            va="center"
        )
        for iso in highlight_iso3
    }

    def update(f):
        f0 = max(frames[0], f - trail_frames)

        trail = df[(df["frame"] >= f0) & (df["frame"] <= f)]
        cur = df[df["frame"] == f]

        trail_scat.set_offsets(trail[["GHG_per_capita", "Vulnerability"]].to_numpy())
        cur_scat.set_offsets(cur[["GHG_per_capita", "Vulnerability"]].to_numpy())

        # Highlight Countries
        hi = cur[cur["ISO3"].isin(highlight_iso3)]
        hi_scat.set_offsets(hi[["GHG_per_capita", "Vulnerability"]].to_numpy())

        for iso, txt in label_texts.items():
            row = hi[hi["ISO3"] == iso]
            if len(row) == 1:
                x = row["GHG_per_capita"].iloc[0]
                y = row["Vulnerability"].iloc[0]
                name = row["Country"].iloc[0]
                txt.set_position((x + 0.8, y))
                txt.set_text(name)
            else:
                txt.set_text("")

        # Titel
        if "t_cont" in cur.columns and len(cur) > 0:
            title.set_text(f"Continuity improvers – t={cur['t_cont'].iloc[0]:.2f}")
        else:
            title.set_text(f"Continuity improvers – frame {f}")

        return [trail_scat, cur_scat, hi_scat, *label_texts.values(), title]

    anim = FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=1000 / fps,
        blit=True,
    )

    anim.save(outfile, fps=fps)
    plt.close(fig)


In [100]:
animate_continuity_improvers(
    interp_df=interp_df,
    countries_continuity=countries_continuity,
    highlight_iso3=["SOM", "QAT", "POL"],
    fps=12,
    trail_frames=40,
    outfile="anim_continuity_improvers_HD.mp4",
)



#### Warum Somalia in der Kontinuitäts-Animation nicht erscheint

Somalia erfuellt zwar den **Kontinuitaets-Filter** (lange zusammenhaengende Phase positiver Jahresveraenderungen) und erscheint daher korrekt in der **Kontinuitaets-Ranking-Tabelle**.

Fuer die **kontinuierliche Animation** ist jedoch zusaetzlich erforderlich, dass fuer ein Land **ausreichend viele aufeinanderfolgende Jahresdaten** vorliegen, um eine Interpolation zwischen den Jahren zu ermoeglichen.

Bei Somalia ist dies nicht der Fall:
- Die Zeitreihe enthaelt **Datenluecken oder sehr kurze Segmente**.
- Dadurch entstehen **keine interpolierbaren Trajektorien**.
- Somalia taucht folglich **nicht im interpolierten DataFrame (`interp_df`)** auf.

Die Animation basiert ausschliesslich auf `interp_df`.
Länder, die dort nicht enthalten sind, **koennen technisch nicht animiert werden**, auch wenn sie in statischen Auswertungen als stabile Improver gelten.

**Wichtig:**
Dies ist **kein Fehler**, sondern eine bewusste methodische Konsequenz:
- Tabellen beantworten: *Welche Länder zeigen nachhaltige Verbesserungsphasen?*
- Animationen beantworten: *Wie verlaufen diese Verbesserungen kontinuierlich im (GHG, Vulnerability)-Raum?*

Somalia erfuellt die erste, aber nicht die zweite Voraussetzung.


## Animation 4 – Globaler Kontext nach Kontinenten (HD, kontinuierlich, geclippt)

**Ziel**
Die globale Dynamik von Ländern im (GHG per capita, Vulnerability)-Raum wird
nach **Kontinenten farblich strukturiert**, um räumliche Muster sichtbar zu machen.

**Warum sinnvoll?**
- Reduziert visuelles Chaos gegenüber Länderfarben.
- Erlaubt direkte Vergleiche zwischen Kontinenten.
- Unterstützt narrative Aussagen wie:
  - „Afrika konzentriert sich im Hoch-Vulnerabilitäts-Bereich“
  - „Europa bewegt sich geschlossen bei niedriger Vulnerabilität“

**Darstellung**
- Jeder Punkt = Land (interpolierte, kontinuierliche Bewegung)
- Farbe = Kontinent
- Trail = letzte Bewegungssegmente (zeitliche Richtung sichtbar)
- Fixierte Achsen (clipped) für Vergleichbarkeit über die Zeit

**Ergebnis**
Eine einordnende Übersicht, die **Makro-Strukturen** sichtbar macht – nicht Einzelfälle.
m

KeyError: ['Continent']